In [1]:
#load modules
import pandas as pd
import scanpy as sc
import numpy as np
import anndata
import os
from collections import Counter
from sklearn.decomposition import PCA


#ploting modules
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import anndata
from scipy import stats
import warnings
import logging
import glob
import os
from upsetplot import UpSet, from_memberships

working_dic="/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/"
os.chdir(working_dic)

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.logging.print_version_and_date()


sc.settings.verbosity = 1
warnings.filterwarnings('ignore')

# If you're using libraries like pandas, you can also set their display options
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Colors for pallettes!
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
dimplot_color=mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#E5E5E5","#FF0000"])

plt.rcParams['pdf.fonttype'] = 42 

Running Scanpy 1.11.3, on 2025-09-22 02:43.


In [18]:
working_dic="/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/"
os.chdir(working_dic)


In [19]:
ls

PCMV_vs_control/  Rejection_vs_control/  Transplant_vs_control/


In [15]:
#!/usr/bin/env python3
import os
import re
import sys
import traceback
import pandas as pd

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs"

# ---- config (thresholds) ----
LOGFC_THRESH = 0.5
PADJ_THRESH = 0.05

# Target column order (exact spellings, per your request)
TARGET_COLS = ['gene', 'p_val', 'avg_log2FC', 'pct. 1', 'pct.2', 'p_val_adj']

def norm_name(s: str) -> str:
    """Normalize a column name for fuzzy matching (lowercase, remove non-alnum)."""
    return re.sub(r'[^a-z0-9]', '', s.lower())

# Map many possible spellings to our target names
NORM_TO_TARGET = {
    # p-values
    'pval': 'p_val',
    'pvalue': 'p_val',
    'p': 'p_val',

    # adjusted p-values
    'pvaladj': 'p_val_adj',
    'padj': 'p_val_adj',
    'padjust': 'p_val_adj',
    'padjusted': 'p_val_adj',
    'padjustment': 'p_val_adj',
    'padjustedvalue': 'p_val_adj',
    'padjustedvalues': 'p_val_adj',
    'padjustedval': 'p_val_adj',
    'pvaladjusted': 'p_val_adj',
    'qval': 'p_val_adj',
    'qvalue': 'p_val_adj',
    'fdr': 'p_val_adj',

    # effect size
    'avglog2fc': 'avg_log2FC',
    'avglogfc': 'avg_log2FC',
    'log2fc': 'avg_log2FC',
    'logfc': 'avg_log2FC',
    'avglog2foldchange': 'avg_log2FC',
    'avglog2_foldchange': 'avg_log2FC',
    'avglog2_fc': 'avg_log2FC',
    'avg_logfc': 'avg_log2FC',  # common Seurat v3 name
    'avglogfc.1': 'avg_log2FC', # occasional export oddities

    # detection fractions
    'pct1': 'pct. 1',   # exactly with a space after the dot (per your request)
    'pct01': 'pct. 1',  # just in case
    'pct_1': 'pct. 1',
    'pct.1': 'pct. 1',
    'pct 1': 'pct. 1',
    'pct2': 'pct.2',
    'pct_2': 'pct.2',
    'pct.2': 'pct.2',
    'pct 2': 'pct.2',

    # gene column
    'gene': 'gene',
    'genes': 'gene',
    'features': 'gene',
    'feature': 'gene',
    'symbol': 'gene',
    'rownames': 'gene',
    'rowname': 'gene',
    'id': 'gene'
}

def infer_and_standardize_columns(df: pd.DataFrame, src_path: str) -> pd.DataFrame:
    """
    Standardize DataFrame to columns:
    ['gene', 'p_val', 'avg_log2FC', 'pct. 1', 'pct.2', 'p_val_adj'].
    Accepts messy headers, unnamed gene column, or gene rownames.
    """
    cols = list(df.columns)
    # If first column looks like an index column name ("Unnamed: 0"), treat as gene
    if cols and re.match(r'^unnamed:?\s*0$', cols[0], flags=re.I):
        df = df.rename(columns={cols[0]: 'gene'})

    # If no obvious gene column but index looks like strings and not useful ints, move it to a column
    if 'gene' not in [c.lower() for c in df.columns]:
        # Detect if index holds gene-like strings (not 0..N-1)
        if not df.index.is_monotonic_increasing or not pd.api.types.is_integer_dtype(df.index):
            df = df.reset_index()
            # After reset, the new column is 'index'
            if 'index' in df.columns:
                df = df.rename(columns={'index': 'gene'})

    # Build a rename map by normalizing column names
    rename_map = {}
    for c in df.columns:
        nc = norm_name(c)
        if nc in NORM_TO_TARGET:
            # Map to our target spelling
            rename_map[c] = NORM_TO_TARGET[nc]
        elif c == 'gene':
            rename_map[c] = 'gene'

    df = df.rename(columns=rename_map)

    # If we still don't have 'gene', but the first column is a good candidate, use it
    if 'gene' not in df.columns and len(df.columns) >= 1:
        first_col = df.columns[0]
        if first_col not in ('p_val', 'avg_log2FC', 'pct. 1', 'pct.2', 'p_val_adj'):
            df = df.rename(columns={first_col: 'gene'})

    # Ensure all target columns exist; if missing, create empty
    for c in TARGET_COLS:
        if c not in df.columns:
            df[c] = pd.NA

    # Keep only target columns, in order
    df = df[TARGET_COLS]

    # Coerce numeric columns (leave 'gene' as is)
    for c in ['p_val', 'avg_log2FC', 'pct. 1', 'pct.2', 'p_val_adj']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Clean up gene strings a bit
    df['gene'] = df['gene'].astype(str).str.strip()

    # Drop rows with no gene (rare but possible)
    df = df[df['gene'].astype(str).str.len() > 0]

    return df

def read_wilcox_file(path: str) -> pd.DataFrame:
    """
    Robust reader for CSV/TSV with auto-detected delimiter and header.
    Tries multiple strategies to cope with messy exports.
    """
    # Try: autodetect delimiter, header=0
    try:
        df = pd.read_csv(path, sep=None, engine='python')
        return df
    except Exception:
        pass

    # Try: explicit common delimiters with header
    for sep in [',', '\t', ';', r'\s+']:
        try:
            df = pd.read_csv(path, sep=sep, engine='python')
            return df
        except Exception:
            continue

    # Try: no header
    try:
        df = pd.read_csv(path, header=None, sep=None, engine='python')
        # If no header, try to set the second through sixth columns to expected names, assume first col = gene
        if df.shape[1] >= 6:
            df.columns = ['gene', 'p_val', 'avg_log2FC', 'pct. 1', 'pct.2', 'p_val_adj'] + \
                         [f'ExtraCol{i}' for i in range(df.shape[1]-6)]
        return df
    except Exception:
        # Last resort: raise
        raise

def process_raw_dir(raw_dir: str):
    comp_dir = os.path.dirname(raw_dir)
    cleaned_dir = os.path.join(comp_dir, "cleaned")
    filtered_dir = os.path.join(comp_dir, "filtered")
    os.makedirs(cleaned_dir, exist_ok=True)
    os.makedirs(filtered_dir, exist_ok=True)

    for fname in os.listdir(raw_dir):
        if not fname.lower().endswith(".csv"):
            continue
        src_path = os.path.join(raw_dir, fname)
        try:
            df_raw = read_wilcox_file(src_path)
            df_clean = infer_and_standardize_columns(df_raw, src_path)

            cleaned_path = os.path.join(cleaned_dir, fname)
            df_clean.to_csv(cleaned_path, index=False)

            # Filtering
            if 'avg_log2FC' in df_clean.columns and 'p_val_adj' in df_clean.columns:
                up = df_clean[(df_clean['avg_log2FC'] > LOGFC_THRESH) & (df_clean['p_val_adj'] < PADJ_THRESH)].copy()
                up['level'] = 'Up-regulated'
                down = df_clean[(df_clean['avg_log2FC'] < -LOGFC_THRESH) & (df_clean['p_val_adj'] < PADJ_THRESH)].copy()
                down['level'] = 'Down-regulated'
                filtered = pd.concat([up, down], axis=0)

                filtered_path = os.path.join(filtered_dir, fname)
                filtered.to_csv(filtered_path, index=False)
            else:
                print(f"[FILTER SKIP] Missing needed columns in {src_path}")

            print(f"✓ Done: {src_path}")
        except Exception as e:
            print(f"[FAIL] {src_path}: {e}")
            traceback.print_exc(file=sys.stdout)

def main():
    if not os.path.isdir(BASE_DIR):
        print(f"Base directory not found: {BASE_DIR}")
        sys.exit(1)

    # Walk every folder named 'raw' under BASE_DIR
    any_found = False
    for dirpath, dirnames, filenames in os.walk(BASE_DIR):
        if os.path.basename(dirpath) == "raw":
            any_found = True
            print(f"Processing: {dirpath}")
            process_raw_dir(dirpath)

    if not any_found:
        print(f"No 'raw' directories found under: {BASE_DIR}")
    else:
        print("All done!")

if __name__ == "__main__":
    main()


Processing: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/Fibroblasts_wilcox_de.csv
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/SMC_wilcox_de.csv
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/Pericyte_wilcox_de.csv
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/Neuronal_cells_wilcox_de.csv
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/CM_wilcox_de.csv
✓ Done: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/raw/Endocardial_Lymphatic_EC_wilcox_de.csv
✓ Done: /da

In [16]:
#!/usr/bin/env python3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs"

# Save plots here inside each comparison directory
VOLCANO_FOLDER_NAME = "volcano_plots"

# Axes/thresholds
PADJ_EPS = 1e-300           # avoid -log10(0)
YLABEL = "-log10(p_adj)"
XLABEL = "log2 Fold Change (avg_log2FC)"

def find_comparisons(base_dir):
    """
    Yield comparison directories that contain both 'cleaned' and 'filtered' subfolders.
    e.g., .../REF_control/PCMV_vs_control, .../REF_rejection/PCMV_vs_rejection
    """
    for dirpath, dirnames, _ in os.walk(base_dir):
        if "cleaned" in dirnames and "filtered" in dirnames:
            yield dirpath

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def plot_volcano_for_file(clean_path, filtered_path, out_dir, title_prefix=""):
    df_all = pd.read_csv(clean_path)

    # Check required columns (from our cleaning step)
    required = {"avg_log2FC", "p_val_adj"}
    if not required.issubset(set(df_all.columns)):
        print(f"[SKIP] Missing columns in {clean_path}. Found: {df_all.columns.tolist()}")
        return

    # Numeric coercions
    df_all["avg_log2FC"] = safe_numeric(df_all["avg_log2FC"])
    df_all["p_val_adj"]  = safe_numeric(df_all["p_val_adj"]).replace(0, PADJ_EPS)
    df_all["neglog10padj"] = -np.log10(df_all["p_val_adj"])

    # Try to read filtered (significant) genes; OK if absent
    df_sig_up = pd.DataFrame(columns=df_all.columns)
    df_sig_down = pd.DataFrame(columns=df_all.columns)
    if os.path.exists(filtered_path):
        df_sig = pd.read_csv(filtered_path)
        if "level" in df_sig.columns:
            # Coerce and compute y for filtered too
            for c in ("avg_log2FC", "p_val_adj"):
                if c in df_sig.columns:
                    df_sig[c] = safe_numeric(df_sig[c])
            df_sig["p_val_adj"] = df_sig["p_val_adj"].replace(0, PADJ_EPS)
            df_sig["neglog10padj"] = -np.log10(df_sig["p_val_adj"])

            df_sig_up = df_sig[df_sig["level"] == "Up-regulated"].copy()
            df_sig_down = df_sig[df_sig["level"] == "Down-regulated"].copy()
        else:
            print(f"[WARN] No 'level' column in {filtered_path}; plotting all genes only.")

    # Cell type name from filename
    celltype = os.path.splitext(os.path.basename(clean_path))[0]

    # Make plot
    plt.figure(figsize=(5, 5))
    # All genes (gray)
    plt.scatter(
        df_all["avg_log2FC"], df_all["neglog10padj"],
        alpha=0.5, s=16, label="All genes", color="gray"
    )
    # Up (red)
    if not df_sig_up.empty:
        plt.scatter(
            df_sig_up["avg_log2FC"], df_sig_up["neglog10padj"],
            alpha=0.9, s=20, label="Up-regulated", color="red"
        )
    # Down (blue)
    if not df_sig_down.empty:
        plt.scatter(
            df_sig_down["avg_log2FC"], df_sig_down["neglog10padj"],
            alpha=0.9, s=20, label="Down-regulated", color="blue"
        )

    # Reference lines
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    # Optional threshold guides (uncomment if you like them)
    # plt.axvline( 0.5, color="black", linestyle=":", linewidth=1)
    # plt.axvline(-0.5, color="black", linestyle=":", linewidth=1)
    # plt.axhline(-np.log10(0.05), color="black", linestyle=":", linewidth=1)

    plt.xlabel(XLABEL)
    plt.ylabel(YLABEL)
    plt.title(f"Volcano: {celltype.replace('_', ' ')}{(' — ' + title_prefix) if title_prefix else ''}")
    plt.legend()
    plt.tight_layout()

    # Save
    os.makedirs(out_dir, exist_ok=True)
    pdf_path = os.path.join(out_dir, f"{celltype}_volcano.pdf")
    plt.savefig(pdf_path, dpi=300)
    plt.close()
    print(f"Saved: {pdf_path}")

def main():
    comps = list(find_comparisons(BASE_DIR))
    if not comps:
        print(f"No comparison directories with cleaned/filtered found under {BASE_DIR}")
        return

    for comp_dir in comps:
        cleaned_dir = os.path.join(comp_dir, "cleaned")
        filtered_dir = os.path.join(comp_dir, "filtered")
        if not (os.path.isdir(cleaned_dir) and os.path.isdir(filtered_dir)):
            continue

        # e.g., REF_control/PCMV_vs_control
        parent = os.path.basename(os.path.dirname(comp_dir))   # REF_control or REF_rejection
        comp_name = os.path.basename(comp_dir)                 # e.g., PCMV_vs_control
        title_prefix = f"{parent}/{comp_name}"

        volcano_dir = os.path.join(comp_dir, VOLCANO_FOLDER_NAME)
        os.makedirs(volcano_dir, exist_ok=True)

        for fname in os.listdir(cleaned_dir):
            if not fname.lower().endswith(".csv"):
                continue
            clean_path = os.path.join(cleaned_dir, fname)
            filtered_path = os.path.join(filtered_dir, fname)  # same filename convention
            try:
                plot_volcano_for_file(clean_path, filtered_path, volcano_dir, title_prefix=title_prefix)
            except Exception as e:
                print(f"[ERROR] {parent}/{comp_name}/{fname}: {e}")

    print("All volcano plots done!")

if __name__ == "__main__":
    main()


Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/Fibroblasts_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/SMC_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/Pericyte_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/Neuronal_cells_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/CM_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/volcano_plots/Endocardial_Lymphatic_EC_wilcox_de_volcano.pdf
Saved: /data2/core-med1/nhadizad

In [17]:
#!/usr/bin/env python3
import os
import pandas as pd
import re

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs"

# Prefix patterns (order matters: pCMV first so it doesn't get caught as Pig)
PATTERNS = [
    (r'^Sus-scrofa---exon-', 'pCMV', ''),   # remove full "Sus-scrofa---exon-" -> pCMV
    (r'^Sus-scrofa---',      'Pig',  ''),   # remove "Sus-scrofa---"          -> Pig
    (r'^Homo-sapiens-?',     'Human',''),   # remove "Homo-sapiens" (with/without '-') -> Human
]

def add_species_and_strip_prefixes(df: pd.DataFrame) -> pd.DataFrame:
    if 'gene' not in df.columns:
        # Try common alternatives just in case
        for alt in ['Gene', 'GENE', 'symbol', 'Symbol']:
            if alt in df.columns:
                df = df.rename(columns={alt: 'gene'})
                break
    if 'gene' not in df.columns:
        raise ValueError("No 'gene' column found to clean.")

    # Ensure string
    df['gene'] = df['gene'].astype(str)

    # Initialize Species as blank
    if 'Species' not in df.columns:
        df['Species'] = ''

    # Apply patterns in order (vectorized)
    for pat, species_label, replacement in PATTERNS:
        mask = df['gene'].str.match(pat, case=True, na=False)
        if mask.any():
            df.loc[mask, 'Species'] = species_label
            df.loc[mask, 'gene'] = df.loc[mask, 'gene'].str.replace(pat, replacement, regex=True)

    # Place Species right after gene
    cols = list(df.columns)
    if 'Species' in cols:
        cols.remove('Species')
        gene_idx = cols.index('gene') if 'gene' in cols else -1
        cols.insert(gene_idx + 1, 'Species')
        df = df[cols]

    return df

def process_folder(in_dir: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    for fname in os.listdir(in_dir):
        if not fname.lower().endswith(".csv"):
            continue
        src = os.path.join(in_dir, fname)
        try:
            df = pd.read_csv(src)
            df = add_species_and_strip_prefixes(df)
            df.to_csv(os.path.join(out_dir, fname), index=False)
            print(f"Saved: {os.path.join(out_dir, fname)}")
        except Exception as e:
            print(f"[FAIL] {src}: {e}")

def main():
    # Walk all comparison dirs under BASE_DIR that contain 'cleaned' or 'filtered'
    for dirpath, dirnames, _ in os.walk(BASE_DIR):
        # Only handle directories that look like comparison roots (siblings of 'raw')
        if 'cleaned' in dirnames or 'filtered' in dirnames:
            comp_dir = dirpath
            prefix_root = os.path.join(comp_dir, "prefix_removed")
            print(f"Processing comparison: {comp_dir}")

            cleaned_in = os.path.join(comp_dir, "cleaned")
            filtered_in = os.path.join(comp_dir, "filtered")
            cleaned_out = os.path.join(prefix_root, "cleaned")
            filtered_out = os.path.join(prefix_root, "filtered")

            if os.path.isdir(cleaned_in):
                process_folder(cleaned_in, cleaned_out)
            else:
                print(f"[INFO] No cleaned/ found in {comp_dir}")

            if os.path.isdir(filtered_in):
                process_folder(filtered_in, filtered_out)
            else:
                print(f"[INFO] No filtered/ found in {comp_dir}")

    print("All done!")

if __name__ == "__main__":
    main()


Processing comparison: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/prefix_removed/cleaned/Fibroblasts_wilcox_de.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/prefix_removed/cleaned/SMC_wilcox_de.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/prefix_removed/cleaned/Pericyte_wilcox_de.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/prefix_removed/cleaned/Neuronal_cells_wilcox_de.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilcox_DEGs/REF_control/Transplant_vs_control/prefix_removed/cleaned/CM_wilcox_de.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Wilc

# Log2FC = 2 pathway enrichment

In [26]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

SOURCE_SUBDIR = "prefix_removed_filtered_05"   # input
TARGET_SUBDIR = os.path.join("enrichment_log2fc_2", "filtered_no_prefix_Log2fc_2")

# Thresholds
LOG2FC_UP = 2
LOG2FC_DOWN = -2
PADJ_MAX = 0.05

# Columns
GENE_COL = "gene"
LOGFC_COL = "avg_log2FC"
PADJ_COL = "p_val_adj"
LEVEL_COL = "level"

def find_source_dirs(base_dir):
    """Yield (group_name, comp_dir, src_dir) for each comparison that has 'prefix_removed_filtered_2'."""
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue

        # comparisons like PCMV_vs_control, Rejection_vs_control, Transplant_vs_control, PCMV_vs_rejection
        for sub in os.listdir(group_path):
            comp_dir = os.path.join(group_path, sub)
            if not os.path.isdir(comp_dir):
                continue
            src_dir = os.path.join(comp_dir, SOURCE_SUBDIR)
            if os.path.isdir(src_dir):
                yield group_name, comp_dir, src_dir

        # edge-case: group root directly contains the folder
        maybe_src = os.path.join(group_path, SOURCE_SUBDIR)
        if os.path.isdir(maybe_src):
            yield group_name, group_path, maybe_src

def loose_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Return rows with |log2FC|>2 and p_adj<0.05 across ALL species; (re)assign 'level'."""
    df = df.copy()
    df[LOGFC_COL] = pd.to_numeric(df[LOGFC_COL], errors="coerce")
    df[PADJ_COL]  = pd.to_numeric(df[PADJ_COL], errors="coerce")

    up   = df[(df[LOGFC_COL] > LOG2FC_UP)    & (df[PADJ_COL] < PADJ_MAX)].copy()
    down = df[(df[LOGFC_COL] < LOG2FC_DOWN) & (df[PADJ_COL] < PADJ_MAX)].copy()

    if not up.empty:
        up[LEVEL_COL] = "Up-regulated"
    if not down.empty:
        down[LEVEL_COL] = "Down-regulated"

    return pd.concat([up, down], ignore_index=True)

def main():
    any_found = False
    for group_name, comp_dir, src_dir in find_source_dirs(BASE_DIR):
        any_found = True
        tgt_dir = os.path.join(comp_dir, TARGET_SUBDIR)
        os.makedirs(tgt_dir, exist_ok=True)
        print(f"[{group_name}] {os.path.basename(comp_dir)} -> {tgt_dir}")

        for fname in os.listdir(src_dir):
            if not fname.lower().endswith(".csv"):
                continue
            fpath = os.path.join(src_dir, fname)
            try:
                df = pd.read_csv(fpath)
                needed = {GENE_COL, LOGFC_COL, PADJ_COL}
                if not needed.issubset(df.columns):
                    print(f"  [SKIP] Missing columns in {fpath}: {df.columns.tolist()}")
                    continue

                df_out = loose_filter(df)
                out_path = os.path.join(tgt_dir, fname)
                df_out.to_csv(out_path, index=False)
                print(f"  Saved: {out_path}  (n={len(df_out)})")
            except Exception as e:
                print(f"  [ERROR] {fpath}: {e}")

    if not any_found:
        print(f"No '{SOURCE_SUBDIR}' folders found under {BASE_DIR}")
    else:
        print("All loose filtered CSVs saved.")

if __name__ == "__main__":
    main()


[REF_Control] Transplant_vs_control -> /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/filtered_no_prefix_Log2fc_2
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/filtered_no_prefix_Log2fc_2/Fibroblasts_wilcox_de.csv  (n=1543)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/filtered_no_prefix_Log2fc_2/SMC_wilcox_de.csv  (n=10)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/filtered_no_prefix_Log2fc_2/Pericyte_wilcox_de.csv  (n=208)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/filtered_no_prefix_Log2fc_2/Neuronal_cells_wilcox_de.csv  (n=110)
  Saved:

In [27]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

GENE_SETS = ["MSigDB_Hallmark_2020"]

LOG2FC_UP = 2.0
LOG2FC_DOWN = -2.0
PADJ_MAX = 0.05

GENE_COL = "gene"
PADJ_COL = "p_val_adj"
LOGFC_COL = "avg_log2FC"
LEVEL_COL = "level"

# Save into each comparison at enrichment_log2fc_2/Results_Hallmark_v1/<celltype>/<gene set>/
RESULTS_ROOT_NAME = os.path.join("enrichment_log2fc_2", "Results_Hallmark_v1")

def find_comparisons(base_dir):
    """
    Yield (group_name, comp_dir, filtered_dir) where filtered_dir is <comparison>/prefix_removed_filtered_05
    """
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue
        for sub in os.listdir(group_path):
            comp_dir = os.path.join(group_path, sub)
            if not os.path.isdir(comp_dir):
                continue
            filtered_dir = os.path.join(comp_dir, "prefix_removed_filtered_05")
            if os.path.isdir(filtered_dir):
                yield (group_name, comp_dir, filtered_dir)
        # edge-case: group root contains the folder
        root_filtered = os.path.join(group_path, "prefix_removed_filtered_05")
        if os.path.isdir(root_filtered):
            yield (group_name, group_path, root_filtered)

def get_regulated_gene_lists(df):
    """
    Apply extra thresholds on ALL genes (no species filter):
      Up:   avg_log2FC > 2  & p_val_adj < 0.05
      Down: avg_log2FC < -2 & p_val_adj < 0.05
    Return up_genes, down_genes.
    """
    for c in [LOGFC_COL, PADJ_COL]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    up_df = df[(df[LOGFC_COL] > LOG2FC_UP) & (df[PADJ_COL] < PADJ_MAX)]
    down_df = df[(df[LOGFC_COL] < LOG2FC_DOWN) & (df[PADJ_COL] < PADJ_MAX)]

    up_genes = (
        up_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in up_df.columns else []
    )
    down_genes = (
        down_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in down_df.columns else []
    )
    return up_genes, down_genes

def run_enrichr(gene_list, gs):
    """Run Enrichr (treating all genes as Human) and return results DF filtered at FDR<0.05."""
    if not gene_list:
        return pd.DataFrame()
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gs,
        organism="Human",    # treat all as Human
        outdir=None,
        cutoff=0.5,
        no_plot=True
    )
    res = enr.results if enr is not None else None
    if res is None or res.empty:
        return pd.DataFrame()
    if "Adjusted P-value" in res.columns:
        res = res[res["Adjusted P-value"] < 0.05].copy()
        res.rename(columns={"Adjusted P-value": "FDR"}, inplace=True)
    elif "FDR" in res.columns:
        res = res[res["FDR"] < 0.05].copy()
    return res

def mirrored_barplot(all_pathways, title, out_pdf):
    if all_pathways.empty:
        return
    col_score = "Combined Score" if "Combined Score" in all_pathways.columns else "Odds Ratio"

    up_df = all_pathways[all_pathways["Regulation"] == "Up-regulated"].sort_values("FDR").head(100)
    down_df = all_pathways[all_pathways["Regulation"] == "Down-regulated"].sort_values("FDR").head(100)

    plot_df = pd.concat([up_df, down_df], ignore_index=True)
    if plot_df.empty:
        return

    plot_df["score_plot"] = plot_df[col_score]
    plot_df.loc[plot_df["Regulation"] == "Down-regulated", "score_plot"] *= -1
    plot_df["Term_label"] = plot_df["Term"] + plot_df["Regulation"].map({"Up-regulated": " ↑", "Down-regulated": " ↓"})
    plot_df = plot_df.sort_values(["Regulation", "FDR"], ascending=[False, True])

    plt.figure(figsize=(13, max(8, int(0.35 * len(plot_df)))))
    colors = plot_df["Regulation"].map({"Up-regulated": "salmon", "Down-regulated": "skyblue"})
    bars = plt.barh(plot_df["Term_label"], plot_df["score_plot"], color=colors)
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("Combined Score (Up=Right, Down=Left)")
    plt.ylabel("Enriched Pathways")
    plt.title(title, fontsize=14)
    plt.tight_layout()

    max_right = plot_df["score_plot"].max() if not plot_df["score_plot"].empty else 1.0
    max_left = abs(plot_df["score_plot"].min()) if not plot_df["score_plot"].empty else 1.0
    for bar, score in zip(bars, plot_df["score_plot"]):
        val = abs(score)
        if score >= 0:
            plt.text(bar.get_width() + 0.01 * max_right, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="left", fontsize=8)
        else:
            plt.text(bar.get_width() - 0.01 * max_left, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="right", fontsize=8)

    plt.savefig(out_pdf, dpi=300)
    plt.close()

def main():
    comparisons = list(find_comparisons(BASE_DIR))
    if not comparisons:
        print(f"No comparison folders with 'prefix_removed_filtered_05' found under {BASE_DIR}")
        return

    for group_name, comp_dir, filtered_dir in comparisons:
        results_root = os.path.join(comp_dir, RESULTS_ROOT_NAME)
        os.makedirs(results_root, exist_ok=True)

        for fname in os.listdir(filtered_dir):
            if not fname.lower().endswith(".csv"):
                continue
            celltype = os.path.splitext(fname)[0]
            fpath = os.path.join(filtered_dir, fname)

            try:
                df = pd.read_csv(fpath)
                needed = {GENE_COL, LOGFC_COL, PADJ_COL}
                if not needed.issubset(df.columns):
                    print(f"[SKIP] Missing required columns in {fpath}. Found: {df.columns.tolist()}")
                    continue

                up_genes, down_genes = get_regulated_gene_lists(df)
                if not up_genes and not down_genes:
                    print(f"[INFO] No genes pass thresholds for {group_name}/{os.path.basename(comp_dir)}/{celltype}")
                    continue

                celltype_dir = os.path.join(results_root, celltype)
                os.makedirs(celltype_dir, exist_ok=True)

                for gs in GENE_SETS:
                    gs_dir = os.path.join(celltype_dir, gs)
                    os.makedirs(gs_dir, exist_ok=True)

                    all_results = []
                    up_res = run_enrichr(up_genes, gs)
                    if not up_res.empty:
                        up_res = up_res.copy()
                        up_res["Regulation"] = "Up-regulated"
                        all_results.append(up_res)

                    down_res = run_enrichr(down_genes, gs)
                    if not down_res.empty:
                        down_res = down_res.copy()
                        down_res["Regulation"] = "Down-regulated"
                        all_results.append(down_res)

                    if not all_results:
                        print(f"[INFO] No significant pathways for {celltype} — {gs}")
                        continue

                    all_pathways = pd.concat(all_results, ignore_index=True)
                    if all_pathways.empty:
                        continue

                    out_csv = os.path.join(gs_dir, "ALL_significant_enrichment_Log2FC_2.csv")
                    all_pathways.to_csv(out_csv, index=False)

                    title = f"Mirrored Enrichment: {celltype} — {group_name}/{os.path.basename(comp_dir)} — {gs}"
                    out_plot = os.path.join(gs_dir, "mirrored_barplot_ALL_enrichment_Log2FC_2.pdf")
                    mirrored_barplot(all_pathways, title, out_plot)

                    print(f"Saved: {out_csv}")
                    print(f"Saved: {out_plot}")

            except Exception as e:
                print(f"[ERROR] {group_name}/{os.path.basename(comp_dir)}/{celltype}: {e}")

    print("All enrichment analyses and plots done!")

if __name__ == "__main__":
    main()


Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_2.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_2.pdf
[INFO] No significant pathways for SMC_wilcox_de — MSigDB_Hallmark_2020
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_2.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/mirrored_barp

# Modified mirrored-barplots

In [28]:
#!/usr/bin/env python3
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------
# Config
# --------------------------------
BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

# We’ll look for results in BOTH thresholds if they exist
RESULT_ROOTS = [
    os.path.join("enrichment_log2fc_2",  "Results_Hallmark_v1"),
    os.path.join("enrichment_log2fc_05", "Results_Hallmark_v1"),
]

# Plot style (from your template)
BAR_WIDTH = 0.7           # bar thickness
PLOT_HEIGHT_FACTOR = 0.06 # vertical size scaling per row
PDF_DPI = 300

# Matplotlib tweaks
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("max_colwidth", None)
plt.rcParams["pdf.fonttype"] = 42   # editable text in Illustrator etc.

# --------------------------------
# Helpers
# --------------------------------
def find_result_dirs(base_dir):
    """
    Yield (group_name, comp_dir, results_root) for each comparison that has
    <comparison>/enrichment_log2fc_*/Results_Hallmark_v1
    """
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue
        for comp in os.listdir(group_path):
            comp_dir = os.path.join(group_path, comp)
            if not os.path.isdir(comp_dir):
                continue
            for rr in RESULT_ROOTS:
                results_root = os.path.join(comp_dir, rr)
                if os.path.isdir(results_root):
                    yield group_name, comp_dir, results_root

def load_all_sig_csvs(gs_dir):
    """
    Return path to a single 'ALL_significant_enrichment_*.csv' if present.
    Prefer the most recently modified if multiple exist.
    """
    candidates = glob.glob(os.path.join(gs_dir, "ALL_significant_enrichment_*.csv"))
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]
    # pick the latest modified to be safe
    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def make_modified_plot(all_pathways: pd.DataFrame, title: str, out_pdf: str):
    if all_pathways.empty:
        return

    # Choose score column
    col_score = "Combined Score" if "Combined Score" in all_pathways.columns else (
        "Odds Ratio" if "Odds Ratio" in all_pathways.columns else None
    )
    if col_score is None:
        print(f"[WARN] No score column found in {out_pdf}. Columns: {all_pathways.columns.tolist()}")
        return

    # Sort Up and Down separately by descending score (longest → shortest)
    up_df = (all_pathways[all_pathways["Regulation"] == "Up-regulated"]
             .sort_values(by=col_score, ascending=False)
             .head(100))
    down_df = (all_pathways[all_pathways["Regulation"] == "Down-regulated"]
               .sort_values(by=col_score, ascending=False)
               .head(100))

    plot_df = pd.concat([up_df, down_df], ignore_index=True)
    if plot_df.empty:
        return

    # Mirror Down to the left
    plot_df["score_plot"] = plot_df[col_score]
    plot_df.loc[plot_df["Regulation"] == "Down-regulated", "score_plot"] *= -1

    # Labels with arrows
    plot_df["Term_label"] = plot_df["Term"] + plot_df["Regulation"].map({
        "Up-regulated": " ↑",
        "Down-regulated": " ↓"
    })

    # Figure size scales with number of rows; show first rows at TOP
    plt.figure(figsize=(7, max(6, int(PLOT_HEIGHT_FACTOR * len(plot_df)))))
    colors = plot_df["Regulation"].map({"Up-regulated": "salmon", "Down-regulated": "skyblue"})

    bars = plt.barh(
        plot_df["Term_label"],
        plot_df["score_plot"],
        color=colors,
        height=BAR_WIDTH,
        edgecolor="black",
        linewidth=0.6,
    )

    plt.gca().invert_yaxis()  # first item appears on top
    plt.axvline(0, color="black", linestyle="-", linewidth=1)
    plt.xlabel(f"{col_score} (Up=Right, Down=Left)")
    plt.ylabel("Enriched Pathways")
    plt.title(title, fontsize=15)
    plt.tight_layout()

    # Save PDF
    plt.savefig(out_pdf, dpi=PDF_DPI)
    plt.close()
    print(f"Saved: {out_pdf}")

def main():
    any_found = False
    for group_name, comp_dir, results_root in find_result_dirs(BASE_DIR):
        any_found = True
        comp_name = os.path.basename(comp_dir)

        # Iterate cell types
        for celltype in os.listdir(results_root):
            celltype_dir = os.path.join(results_root, celltype)
            if not os.path.isdir(celltype_dir):
                continue

            # Iterate gene set folders (e.g., MSigDB_Hallmark_2020)
            for gs in os.listdir(celltype_dir):
                gs_dir = os.path.join(celltype_dir, gs)
                if not os.path.isdir(gs_dir):
                    continue

                csv_path = load_all_sig_csvs(gs_dir)
                if not csv_path:
                    continue

                try:
                    df = pd.read_csv(csv_path)
                    if df.empty:
                        continue
                    if "Regulation" not in df.columns or "Term" not in df.columns:
                        print(f"[SKIP] Missing required columns in {csv_path}")
                        continue

                    # Title & output path
                    title = f"Mirrored Enrichment: {celltype} — {group_name}/{comp_name} — {gs}"
                    # Put 'modified' at the end of the file name
                    out_pdf = os.path.join(
                        gs_dir,
                        os.path.splitext(os.path.basename(csv_path))[0]
                        .replace("ALL_significant_enrichment", "mirrored_barplot_ALL_enrichment")
                        + "_modified.pdf"
                    )

                    make_modified_plot(df, title, out_pdf)

                except Exception as e:
                    print(f"[ERROR] {csv_path}: {e}")

    if not any_found:
        print(f"No Results_Hallmark_v1 directories found under {BASE_DIR}")
    else:
        print("All modified barplots generated.")

if __name__ == "__main__":
    main()


Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_2_modified.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/EC_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_2_modified.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Neuronal_cells_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_2_modified.pdf
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_2/Results_Hallmark_v1/Endocardial_Lymphatic_EC_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log

# Log2FC = 1 enrichment

In [2]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

SOURCE_SUBDIR = "prefix_removed_filtered_05"   # input
TARGET_SUBDIR = os.path.join("enrichment_log2fc_1", "filtered_no_prefix_Log2fc_1")

# Thresholds
LOG2FC_UP = 1
LOG2FC_DOWN = -1
PADJ_MAX = 0.05

# Columns
GENE_COL = "gene"
LOGFC_COL = "avg_log2FC"
PADJ_COL = "p_val_adj"
LEVEL_COL = "level"

def find_source_dirs(base_dir):
    """Yield (group_name, comp_dir, src_dir) for each comparison that has 'prefix_removed_filtered_1'."""
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue

        # comparisons like PCMV_vs_control, Rejection_vs_control, Transplant_vs_control, PCMV_vs_rejection
        for sub in os.listdir(group_path):
            comp_dir = os.path.join(group_path, sub)
            if not os.path.isdir(comp_dir):
                continue
            src_dir = os.path.join(comp_dir, SOURCE_SUBDIR)
            if os.path.isdir(src_dir):
                yield group_name, comp_dir, src_dir

        # edge-case: group root directly contains the folder
        maybe_src = os.path.join(group_path, SOURCE_SUBDIR)
        if os.path.isdir(maybe_src):
            yield group_name, group_path, maybe_src

def loose_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Return rows with |log2FC|>2 and p_adj<0.05 across ALL species; (re)assign 'level'."""
    df = df.copy()
    df[LOGFC_COL] = pd.to_numeric(df[LOGFC_COL], errors="coerce")
    df[PADJ_COL]  = pd.to_numeric(df[PADJ_COL], errors="coerce")

    up   = df[(df[LOGFC_COL] > LOG2FC_UP)    & (df[PADJ_COL] < PADJ_MAX)].copy()
    down = df[(df[LOGFC_COL] < LOG2FC_DOWN) & (df[PADJ_COL] < PADJ_MAX)].copy()

    if not up.empty:
        up[LEVEL_COL] = "Up-regulated"
    if not down.empty:
        down[LEVEL_COL] = "Down-regulated"

    return pd.concat([up, down], ignore_index=True)

def main():
    any_found = False
    for group_name, comp_dir, src_dir in find_source_dirs(BASE_DIR):
        any_found = True
        tgt_dir = os.path.join(comp_dir, TARGET_SUBDIR)
        os.makedirs(tgt_dir, exist_ok=True)
        print(f"[{group_name}] {os.path.basename(comp_dir)} -> {tgt_dir}")

        for fname in os.listdir(src_dir):
            if not fname.lower().endswith(".csv"):
                continue
            fpath = os.path.join(src_dir, fname)
            try:
                df = pd.read_csv(fpath)
                needed = {GENE_COL, LOGFC_COL, PADJ_COL}
                if not needed.issubset(df.columns):
                    print(f"  [SKIP] Missing columns in {fpath}: {df.columns.tolist()}")
                    continue

                df_out = loose_filter(df)
                out_path = os.path.join(tgt_dir, fname)
                df_out.to_csv(out_path, index=False)
                print(f"  Saved: {out_path}  (n={len(df_out)})")
            except Exception as e:
                print(f"  [ERROR] {fpath}: {e}")

    if not any_found:
        print(f"No '{SOURCE_SUBDIR}' folders found under {BASE_DIR}")
    else:
        print("All loose filtered CSVs saved.")

if __name__ == "__main__":
    main()


[REF_Control] Transplant_vs_control -> /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/filtered_no_prefix_Log2fc_1
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/filtered_no_prefix_Log2fc_1/Fibroblasts_wilcox_de.csv  (n=2887)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/filtered_no_prefix_Log2fc_1/SMC_wilcox_de.csv  (n=10)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/filtered_no_prefix_Log2fc_1/Pericyte_wilcox_de.csv  (n=360)
  Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/filtered_no_prefix_Log2fc_1/Neuronal_cells_wilcox_de.csv  (n=145)
  Saved:

In [3]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

GENE_SETS = ["MSigDB_Hallmark_2020"]

LOG2FC_UP = 1.0
LOG2FC_DOWN = -1.0
PADJ_MAX = 0.05

GENE_COL = "gene"
PADJ_COL = "p_val_adj"
LOGFC_COL = "avg_log2FC"
LEVEL_COL = "level"

# Save into each comparison at enrichment_log2fc_2/Results_Hallmark_v1/<celltype>/<gene set>/
RESULTS_ROOT_NAME = os.path.join("enrichment_log2fc_1", "Results_Hallmark_v1")

def find_comparisons(base_dir):
    """
    Yield (group_name, comp_dir, filtered_dir) where filtered_dir is <comparison>/prefix_removed_filtered_05
    """
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue
        for sub in os.listdir(group_path):
            comp_dir = os.path.join(group_path, sub)
            if not os.path.isdir(comp_dir):
                continue
            filtered_dir = os.path.join(comp_dir, "prefix_removed_filtered_05")
            if os.path.isdir(filtered_dir):
                yield (group_name, comp_dir, filtered_dir)
        # edge-case: group root contains the folder
        root_filtered = os.path.join(group_path, "prefix_removed_filtered_05")
        if os.path.isdir(root_filtered):
            yield (group_name, group_path, root_filtered)

def get_regulated_gene_lists(df):
    """
    Apply extra thresholds on ALL genes (no species filter):
      Up:   avg_log2FC > 1  & p_val_adj < 0.05
      Down: avg_log2FC < -1 & p_val_adj < 0.05
    Return up_genes, down_genes.
    """
    for c in [LOGFC_COL, PADJ_COL]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    up_df = df[(df[LOGFC_COL] > LOG2FC_UP) & (df[PADJ_COL] < PADJ_MAX)]
    down_df = df[(df[LOGFC_COL] < LOG2FC_DOWN) & (df[PADJ_COL] < PADJ_MAX)]

    up_genes = (
        up_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in up_df.columns else []
    )
    down_genes = (
        down_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in down_df.columns else []
    )
    return up_genes, down_genes

def run_enrichr(gene_list, gs):
    """Run Enrichr (treating all genes as Human) and return results DF filtered at FDR<0.05."""
    if not gene_list:
        return pd.DataFrame()
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gs,
        organism="Human",    # treat all as Human
        outdir=None,
        cutoff=0.5,
        no_plot=True
    )
    res = enr.results if enr is not None else None
    if res is None or res.empty:
        return pd.DataFrame()
    if "Adjusted P-value" in res.columns:
        res = res[res["Adjusted P-value"] < 0.05].copy()
        res.rename(columns={"Adjusted P-value": "FDR"}, inplace=True)
    elif "FDR" in res.columns:
        res = res[res["FDR"] < 0.05].copy()
    return res

def mirrored_barplot(all_pathways, title, out_pdf):
    if all_pathways.empty:
        return
    col_score = "Combined Score" if "Combined Score" in all_pathways.columns else "Odds Ratio"

    up_df = all_pathways[all_pathways["Regulation"] == "Up-regulated"].sort_values("FDR").head(100)
    down_df = all_pathways[all_pathways["Regulation"] == "Down-regulated"].sort_values("FDR").head(100)

    plot_df = pd.concat([up_df, down_df], ignore_index=True)
    if plot_df.empty:
        return

    plot_df["score_plot"] = plot_df[col_score]
    plot_df.loc[plot_df["Regulation"] == "Down-regulated", "score_plot"] *= -1
    plot_df["Term_label"] = plot_df["Term"] + plot_df["Regulation"].map({"Up-regulated": " ↑", "Down-regulated": " ↓"})
    plot_df = plot_df.sort_values(["Regulation", "FDR"], ascending=[False, True])

    plt.figure(figsize=(13, max(8, int(0.35 * len(plot_df)))))
    colors = plot_df["Regulation"].map({"Up-regulated": "salmon", "Down-regulated": "skyblue"})
    bars = plt.barh(plot_df["Term_label"], plot_df["score_plot"], color=colors)
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("Combined Score (Up=Right, Down=Left)")
    plt.ylabel("Enriched Pathways")
    plt.title(title, fontsize=14)
    plt.tight_layout()

    max_right = plot_df["score_plot"].max() if not plot_df["score_plot"].empty else 1.0
    max_left = abs(plot_df["score_plot"].min()) if not plot_df["score_plot"].empty else 1.0
    for bar, score in zip(bars, plot_df["score_plot"]):
        val = abs(score)
        if score >= 0:
            plt.text(bar.get_width() + 0.01 * max_right, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="left", fontsize=8)
        else:
            plt.text(bar.get_width() - 0.01 * max_left, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="right", fontsize=8)

    plt.savefig(out_pdf, dpi=300)
    plt.close()

def main():
    comparisons = list(find_comparisons(BASE_DIR))
    if not comparisons:
        print(f"No comparison folders with 'prefix_removed_filtered_05' found under {BASE_DIR}")
        return

    for group_name, comp_dir, filtered_dir in comparisons:
        results_root = os.path.join(comp_dir, RESULTS_ROOT_NAME)
        os.makedirs(results_root, exist_ok=True)

        for fname in os.listdir(filtered_dir):
            if not fname.lower().endswith(".csv"):
                continue
            celltype = os.path.splitext(fname)[0]
            fpath = os.path.join(filtered_dir, fname)

            try:
                df = pd.read_csv(fpath)
                needed = {GENE_COL, LOGFC_COL, PADJ_COL}
                if not needed.issubset(df.columns):
                    print(f"[SKIP] Missing required columns in {fpath}. Found: {df.columns.tolist()}")
                    continue

                up_genes, down_genes = get_regulated_gene_lists(df)
                if not up_genes and not down_genes:
                    print(f"[INFO] No genes pass thresholds for {group_name}/{os.path.basename(comp_dir)}/{celltype}")
                    continue

                celltype_dir = os.path.join(results_root, celltype)
                os.makedirs(celltype_dir, exist_ok=True)

                for gs in GENE_SETS:
                    gs_dir = os.path.join(celltype_dir, gs)
                    os.makedirs(gs_dir, exist_ok=True)

                    all_results = []
                    up_res = run_enrichr(up_genes, gs)
                    if not up_res.empty:
                        up_res = up_res.copy()
                        up_res["Regulation"] = "Up-regulated"
                        all_results.append(up_res)

                    down_res = run_enrichr(down_genes, gs)
                    if not down_res.empty:
                        down_res = down_res.copy()
                        down_res["Regulation"] = "Down-regulated"
                        all_results.append(down_res)

                    if not all_results:
                        print(f"[INFO] No significant pathways for {celltype} — {gs}")
                        continue

                    all_pathways = pd.concat(all_results, ignore_index=True)
                    if all_pathways.empty:
                        continue

                    out_csv = os.path.join(gs_dir, "ALL_significant_enrichment_Log2FC_1.csv")
                    all_pathways.to_csv(out_csv, index=False)

                    title = f"Mirrored Enrichment: {celltype} — {group_name}/{os.path.basename(comp_dir)} — {gs}"
                    out_plot = os.path.join(gs_dir, "mirrored_barplot_ALL_enrichment_Log2FC_1.pdf")
                    mirrored_barplot(all_pathways, title, out_plot)

                    print(f"Saved: {out_csv}")
                    print(f"Saved: {out_plot}")

            except Exception as e:
                print(f"[ERROR] {group_name}/{os.path.basename(comp_dir)}/{celltype}: {e}")

    print("All enrichment analyses and plots done!")

if __name__ == "__main__":
    main()


Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_1.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_1.pdf
[INFO] No significant pathways for SMC_wilcox_de — MSigDB_Hallmark_2020
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_1.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_1/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/mirrored_barp

# Log2fc = 0.5 enrichment

In [4]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt

BASE_DIR = "/data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1"

GENE_SETS = ["MSigDB_Hallmark_2020"]

LOG2FC_UP = 0.5
LOG2FC_DOWN = -0.5
PADJ_MAX = 0.05

GENE_COL = "gene"
PADJ_COL = "p_val_adj"
LOGFC_COL = "avg_log2FC"
LEVEL_COL = "level"

# Save into each comparison at enrichment_log2fc_2/Results_Hallmark_v1/<celltype>/<gene set>/
RESULTS_ROOT_NAME = os.path.join("enrichment_log2fc_05", "Results_Hallmark_v1")

def find_comparisons(base_dir):
    """
    Yield (group_name, comp_dir, filtered_dir) where filtered_dir is <comparison>/prefix_removed_filtered_05
    """
    for group_name in ["REF_Control", "REF_Rejection"]:
        group_path = os.path.join(base_dir, group_name)
        if not os.path.isdir(group_path):
            continue
        for sub in os.listdir(group_path):
            comp_dir = os.path.join(group_path, sub)
            if not os.path.isdir(comp_dir):
                continue
            filtered_dir = os.path.join(comp_dir, "prefix_removed_filtered_05")
            if os.path.isdir(filtered_dir):
                yield (group_name, comp_dir, filtered_dir)
        # edge-case: group root contains the folder
        root_filtered = os.path.join(group_path, "prefix_removed_filtered_05")
        if os.path.isdir(root_filtered):
            yield (group_name, group_path, root_filtered)

def get_regulated_gene_lists(df):
    """
    Apply extra thresholds on ALL genes (no species filter):
      Up:   avg_log2FC > 0.5  & p_val_adj < 0.05
      Down: avg_log2FC < -0.5 & p_val_adj < 0.05
    Return up_genes, down_genes.
    """
    for c in [LOGFC_COL, PADJ_COL]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    up_df = df[(df[LOGFC_COL] > LOG2FC_UP) & (df[PADJ_COL] < PADJ_MAX)]
    down_df = df[(df[LOGFC_COL] < LOG2FC_DOWN) & (df[PADJ_COL] < PADJ_MAX)]

    up_genes = (
        up_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in up_df.columns else []
    )
    down_genes = (
        down_df[GENE_COL].dropna().astype(str).str.strip().replace("", np.nan).dropna().unique().tolist()
        if GENE_COL in down_df.columns else []
    )
    return up_genes, down_genes

def run_enrichr(gene_list, gs):
    """Run Enrichr (treating all genes as Human) and return results DF filtered at FDR<0.05."""
    if not gene_list:
        return pd.DataFrame()
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=gs,
        organism="Human",    # treat all as Human
        outdir=None,
        cutoff=0.5,
        no_plot=True
    )
    res = enr.results if enr is not None else None
    if res is None or res.empty:
        return pd.DataFrame()
    if "Adjusted P-value" in res.columns:
        res = res[res["Adjusted P-value"] < 0.05].copy()
        res.rename(columns={"Adjusted P-value": "FDR"}, inplace=True)
    elif "FDR" in res.columns:
        res = res[res["FDR"] < 0.05].copy()
    return res

def mirrored_barplot(all_pathways, title, out_pdf):
    if all_pathways.empty:
        return
    col_score = "Combined Score" if "Combined Score" in all_pathways.columns else "Odds Ratio"

    up_df = all_pathways[all_pathways["Regulation"] == "Up-regulated"].sort_values("FDR").head(100)
    down_df = all_pathways[all_pathways["Regulation"] == "Down-regulated"].sort_values("FDR").head(100)

    plot_df = pd.concat([up_df, down_df], ignore_index=True)
    if plot_df.empty:
        return

    plot_df["score_plot"] = plot_df[col_score]
    plot_df.loc[plot_df["Regulation"] == "Down-regulated", "score_plot"] *= -1
    plot_df["Term_label"] = plot_df["Term"] + plot_df["Regulation"].map({"Up-regulated": " ↑", "Down-regulated": " ↓"})
    plot_df = plot_df.sort_values(["Regulation", "FDR"], ascending=[False, True])

    plt.figure(figsize=(13, max(8, int(0.35 * len(plot_df)))))
    colors = plot_df["Regulation"].map({"Up-regulated": "salmon", "Down-regulated": "skyblue"})
    bars = plt.barh(plot_df["Term_label"], plot_df["score_plot"], color=colors)
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("Combined Score (Up=Right, Down=Left)")
    plt.ylabel("Enriched Pathways")
    plt.title(title, fontsize=14)
    plt.tight_layout()

    max_right = plot_df["score_plot"].max() if not plot_df["score_plot"].empty else 1.0
    max_left = abs(plot_df["score_plot"].min()) if not plot_df["score_plot"].empty else 1.0
    for bar, score in zip(bars, plot_df["score_plot"]):
        val = abs(score)
        if score >= 0:
            plt.text(bar.get_width() + 0.01 * max_right, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="left", fontsize=8)
        else:
            plt.text(bar.get_width() - 0.01 * max_left, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                     va="center", ha="right", fontsize=8)

    plt.savefig(out_pdf, dpi=300)
    plt.close()

def main():
    comparisons = list(find_comparisons(BASE_DIR))
    if not comparisons:
        print(f"No comparison folders with 'prefix_removed_filtered_05' found under {BASE_DIR}")
        return

    for group_name, comp_dir, filtered_dir in comparisons:
        results_root = os.path.join(comp_dir, RESULTS_ROOT_NAME)
        os.makedirs(results_root, exist_ok=True)

        for fname in os.listdir(filtered_dir):
            if not fname.lower().endswith(".csv"):
                continue
            celltype = os.path.splitext(fname)[0]
            fpath = os.path.join(filtered_dir, fname)

            try:
                df = pd.read_csv(fpath)
                needed = {GENE_COL, LOGFC_COL, PADJ_COL}
                if not needed.issubset(df.columns):
                    print(f"[SKIP] Missing required columns in {fpath}. Found: {df.columns.tolist()}")
                    continue

                up_genes, down_genes = get_regulated_gene_lists(df)
                if not up_genes and not down_genes:
                    print(f"[INFO] No genes pass thresholds for {group_name}/{os.path.basename(comp_dir)}/{celltype}")
                    continue

                celltype_dir = os.path.join(results_root, celltype)
                os.makedirs(celltype_dir, exist_ok=True)

                for gs in GENE_SETS:
                    gs_dir = os.path.join(celltype_dir, gs)
                    os.makedirs(gs_dir, exist_ok=True)

                    all_results = []
                    up_res = run_enrichr(up_genes, gs)
                    if not up_res.empty:
                        up_res = up_res.copy()
                        up_res["Regulation"] = "Up-regulated"
                        all_results.append(up_res)

                    down_res = run_enrichr(down_genes, gs)
                    if not down_res.empty:
                        down_res = down_res.copy()
                        down_res["Regulation"] = "Down-regulated"
                        all_results.append(down_res)

                    if not all_results:
                        print(f"[INFO] No significant pathways for {celltype} — {gs}")
                        continue

                    all_pathways = pd.concat(all_results, ignore_index=True)
                    if all_pathways.empty:
                        continue

                    out_csv = os.path.join(gs_dir, "ALL_significant_enrichment_Log2FC_05.csv")
                    all_pathways.to_csv(out_csv, index=False)

                    title = f"Mirrored Enrichment: {celltype} — {group_name}/{os.path.basename(comp_dir)} — {gs}"
                    out_plot = os.path.join(gs_dir, "mirrored_barplot_ALL_enrichment_Log2FC_05.pdf")
                    mirrored_barplot(all_pathways, title, out_plot)

                    print(f"Saved: {out_csv}")
                    print(f"Saved: {out_plot}")

            except Exception as e:
                print(f"[ERROR] {group_name}/{os.path.basename(comp_dir)}/{celltype}: {e}")

    print("All enrichment analyses and plots done!")

if __name__ == "__main__":
    main()


Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_05/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_05.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_05/Results_Hallmark_v1/Fibroblasts_wilcox_de/MSigDB_Hallmark_2020/mirrored_barplot_ALL_enrichment_Log2FC_05.pdf
[INFO] No significant pathways for SMC_wilcox_de — MSigDB_Hallmark_2020
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_05/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/ALL_significant_enrichment_Log2FC_05.csv
Saved: /data2/core-med1/nhadizad/nastaran/Xenotransplant/Human_xeno/Enrichment_v1/REF_Control/Transplant_vs_control/enrichment_log2fc_05/Results_Hallmark_v1/Pericyte_wilcox_de/MSigDB_Hallmark_2020/mirror